# Laboratório — Softmax e cross-entropy multiclasse

Implementaremos softmax, log-sum-exp, log-softmax e cross-entropy em **NumPy puro**.
O foco é o valor da loss e o gradiente local $\mathbf{p}-\mathbf{q}$; não há
backpropagation pela rede, autograd, PyTorch, TensorFlow ou JAX.

Este é o artefato executável da Aula 07 do M5 — Redes Neurais do Zero.


## Goal

Ao final, teremos evidência executável de que:

1. softmax produz distribuições por linha;
2. log-sum-exp e cross-entropy permanecem finitos em logits extremos;
3. rótulos esparsos e one-hot produzem a mesma loss;
4. deslocar logits por uma constante não altera probabilidades nem loss;
5. permutar classes e rótulos de forma consistente preserva a loss;
6. normalizar no eixo do lote é detectável;
7. o gradiente analítico $\mathbf{p}-\mathbf{q}$ coincide com diferenças centrais;
8. a redução média divide o gradiente pelo tamanho do lote.


## Setup

- Python >= 3.11
- NumPy >= 1.26
- Matplotlib >= 3.8
- nbformat >= 5.9 apenas para validar o arquivo

Dados: matrizes explícitas e logits sintéticos determinísticos. Não há download,
credencial, dataset externo ou estado oculto. Seed: `20260909`; dtype: `float64`.


In [ ]:
import platform

import matplotlib.pyplot as plt
import numpy as np

SEED = 20260909
DTYPE = np.float64
rng = np.random.default_rng(SEED)

print({
    "python": platform.python_version(),
    "numpy": np.__version__,
    "matplotlib": plt.matplotlib.__version__,
    "seed": SEED,
    "dtype": str(np.dtype(DTYPE)),
})


## Steps

### 1. Contrato dos logits e reduções

Exemplos ficam nas linhas e classes nas colunas: `(m, C)`, com `C >= 2`.
A redução acontece somente depois de obter uma loss por exemplo.


In [ ]:
class MulticlassContractError(ValueError):
    pass


def validate_logits(logits):
    logits = np.asarray(logits, dtype=DTYPE)
    if logits.ndim != 2:
        raise MulticlassContractError(
            f"logits devem ter shape (m, C); recebido {logits.shape}"
        )
    if logits.shape[0] < 1 or logits.shape[1] < 2:
        raise MulticlassContractError("são exigidos m >= 1 e C >= 2")
    if not np.isfinite(logits).all():
        raise FloatingPointError("logits devem ser finitos")
    return logits


def reduce_examples(losses, reduction="mean"):
    losses = np.asarray(losses, dtype=DTYPE)
    if losses.ndim != 1 or losses.size == 0 or not np.isfinite(losses).all():
        raise MulticlassContractError("losses devem formar um vetor finito não vazio")
    if reduction == "none":
        return losses.copy()
    if reduction == "sum":
        return float(np.sum(losses, dtype=DTYPE))
    if reduction == "mean":
        return float(np.mean(losses, dtype=DTYPE))
    raise MulticlassContractError("reduction deve ser 'none', 'sum' ou 'mean'")


### 2. Log-sum-exp, log-softmax e softmax estáveis

Para cada linha, subtraímos o máximo antes da exponencial:

$$
\operatorname{LSE}(\mathbf z)=a+\log\sum_j e^{z_j-a},
\qquad a=\max_j z_j.
$$


In [ ]:
def logsumexp_rows(logits):
    logits = validate_logits(logits)
    maximum = np.max(logits, axis=1, keepdims=True)
    shifted = logits - maximum
    result = maximum + np.log(np.sum(np.exp(shifted), axis=1, keepdims=True))
    if result.shape != (logits.shape[0], 1) or not np.isfinite(result).all():
        raise FloatingPointError("log-sum-exp violou shape ou finitude")
    return result


def log_softmax(logits):
    logits = validate_logits(logits)
    result = logits - logsumexp_rows(logits)
    if result.shape != logits.shape or not np.isfinite(result).all():
        raise FloatingPointError("log-softmax violou shape ou finitude")
    return result


def softmax(logits):
    logits = validate_logits(logits)
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp_shifted = np.exp(shifted)
    probabilities = exp_shifted / np.sum(exp_shifted, axis=1, keepdims=True)
    if not np.allclose(probabilities.sum(axis=1), 1.0, atol=2e-15, rtol=0.0):
        raise FloatingPointError("probabilidades não somaram 1 por linha")
    return probabilities


### 3. Exemplo manual `[2, 1, 0]`

O máximo é 2. As exponenciais deslocadas são `[1, exp(-1), exp(-2)]` e a
probabilidade esperada é aproximadamente `[0.665241, 0.244728, 0.090031]`.


In [ ]:
manual_logits = np.array([[2.0, 1.0, 0.0]])
manual_shifted_exp = np.exp(manual_logits - manual_logits.max(axis=1, keepdims=True))
manual_probabilities = softmax(manual_logits)
manual_expected = np.array([[0.6652409557748219,
                             0.24472847105479764,
                             0.09003057317038046]])

print("exponenciais deslocadas:", np.round(manual_shifted_exp, 9))
print("probabilidades:", np.round(manual_probabilities, 9))
print("soma da linha:", manual_probabilities.sum(axis=1))


### 4. Cross-entropy com rótulo esparso

Para índices inteiros `labels.shape == (m,)`, calculamos

$$
\ell_i=\operatorname{LSE}(\mathbf z_i)-z_{i,y_i}.
$$


In [ ]:
def validate_sparse_labels(labels, batch_size, class_count):
    labels = np.asarray(labels)
    if labels.shape != (batch_size,):
        raise MulticlassContractError(
            f"labels esparsos devem ter shape {(batch_size,)}; recebido {labels.shape}"
        )
    if not np.issubdtype(labels.dtype, np.integer):
        raise MulticlassContractError("labels esparsos devem ter dtype inteiro")
    if np.any((labels < 0) | (labels >= class_count)):
        raise MulticlassContractError("índice de classe fora de [0, C-1]")
    return labels.astype(np.int64, copy=False)


def cross_entropy_sparse(logits, labels, reduction="mean"):
    logits = validate_logits(logits)
    labels = validate_sparse_labels(labels, logits.shape[0], logits.shape[1])
    rows = np.arange(logits.shape[0])
    losses = logsumexp_rows(logits)[:, 0] - logits[rows, labels]
    return reduce_examples(losses, reduction)


manual_loss_class_0 = cross_entropy_sparse(manual_logits, np.array([0]))
manual_loss_class_2 = cross_entropy_sparse(manual_logits, np.array([2]))
print({
    "loss_classe_0": manual_loss_class_0,
    "loss_classe_2": manual_loss_class_2,
})


### 5. Cross-entropy com distribuição-alvo

Alvos densos têm o mesmo shape dos logits, valores não negativos e soma 1 por linha.
Não normalizamos entradas inválidas silenciosamente.


In [ ]:
def validate_dense_targets(targets, expected_shape):
    targets = np.asarray(targets, dtype=DTYPE)
    if targets.shape != expected_shape:
        raise MulticlassContractError(
            f"alvo denso deve ter shape {expected_shape}; recebido {targets.shape}"
        )
    if not np.isfinite(targets).all() or np.any(targets < 0.0):
        raise MulticlassContractError("alvo denso deve ser finito e não negativo")
    if not np.allclose(targets.sum(axis=1), 1.0, atol=1e-12, rtol=0.0):
        raise MulticlassContractError("cada linha do alvo denso deve somar 1")
    return targets


def cross_entropy_dense(logits, targets, reduction="mean"):
    logits = validate_logits(logits)
    targets = validate_dense_targets(targets, logits.shape)
    losses = -np.sum(targets * log_softmax(logits), axis=1)
    return reduce_examples(losses, reduction)


def one_hot(labels, class_count):
    labels = validate_sparse_labels(labels, len(labels), class_count)
    result = np.zeros((labels.size, class_count), dtype=DTYPE)
    result[np.arange(labels.size), labels] = 1.0
    return result


batch_logits = np.array([[2.0, 1.0, 0.0],
                         [-1.0, 0.5, 1.5],
                         [0.2, -0.3, 0.1]])
batch_labels = np.array([0, 2, 1])
batch_one_hot = one_hot(batch_labels, class_count=3)
sparse_losses = cross_entropy_sparse(batch_logits, batch_labels, "none")
dense_losses = cross_entropy_dense(batch_logits, batch_one_hot, "none")
one_hot_equivalence_error = float(np.max(np.abs(sparse_losses - dense_losses)))

print("losses esparsas:", np.round(sparse_losses, 9))
print(f"erro esparso vs one-hot={one_hot_equivalence_error:.3e}")


### 6. Baseline uniforme e reduções

Com logits iguais, cada classe recebe probabilidade $1/C$ e a loss vale $\log C$.
Também conferimos a identidade `sum == mean * m`.


In [ ]:
class_count = 5
uniform_logits = np.zeros((7, class_count), dtype=DTYPE)
uniform_labels = np.arange(7) % class_count
uniform_none = cross_entropy_sparse(uniform_logits, uniform_labels, "none")
uniform_sum = cross_entropy_sparse(uniform_logits, uniform_labels, "sum")
uniform_mean = cross_entropy_sparse(uniform_logits, uniform_labels, "mean")

print({
    "classes": class_count,
    "loss_uniforme": uniform_mean,
    "log_C": float(np.log(class_count)),
    "soma": uniform_sum,
})


### 7. Invariância a deslocamentos por linha

Cada exemplo pode receber uma constante diferente. Como a mesma constante é aplicada
a todas as suas classes, probabilidades e cross-entropy devem permanecer iguais.


In [ ]:
row_shifts = np.array([[1e6], [-1e6], [12345.0]])
shifted_logits = batch_logits + row_shifts
probability_shift_error = float(np.max(np.abs(
    softmax(shifted_logits) - softmax(batch_logits)
)))
loss_shift_error = float(abs(
    cross_entropy_sparse(shifted_logits, batch_labels)
    - cross_entropy_sparse(batch_logits, batch_labels)
))

print({
    "erro_probabilidades": probability_shift_error,
    "erro_loss": loss_shift_error,
})


### 8. Permutação consistente das classes

Se reordenamos colunas e remapeamos os índices, mudamos apenas os nomes das classes,
não a loss. Esta verificação protege o contrato índice–rótulo.


In [ ]:
permutation = np.array([2, 0, 1])
permuted_logits = batch_logits[:, permutation]
inverse_permutation = np.empty_like(permutation)
inverse_permutation[permutation] = np.arange(permutation.size)
permuted_labels = inverse_permutation[batch_labels]

original_loss = cross_entropy_sparse(batch_logits, batch_labels)
permuted_loss = cross_entropy_sparse(permuted_logits, permuted_labels)
permutation_error = float(abs(original_loss - permuted_loss))

print({"loss_original": original_loss,
       "loss_permutada": permuted_loss,
       "erro": permutation_error})


### 9. Extremos: a forma ingênua quebra

`exp(1000)` transborda em `float64`. A implementação deslocada permanece finita e
produz perdas coerentes para classes corretas e incorretas.


In [ ]:
extreme_logits = np.array([[1000.0, 0.0, -1000.0],
                           [1000.0, 0.0, -1000.0]])
extreme_labels = np.array([0, 2])

with np.errstate(over="ignore", divide="ignore", invalid="ignore"):
    naive_exp = np.exp(extreme_logits)
    naive_probabilities = naive_exp / naive_exp.sum(axis=1, keepdims=True)
    naive_losses = -np.log(naive_probabilities[
        np.arange(extreme_labels.size), extreme_labels
    ])

stable_probabilities = softmax(extreme_logits)
stable_losses = cross_entropy_sparse(extreme_logits, extreme_labels, "none")

print("softmax ingênua:\n", naive_probabilities)
print("loss ingênua:", naive_losses)
print("softmax estável:\n", stable_probabilities)
print("loss estável:", stable_losses)


### 10. Contraprova do eixo incorreto

Normalizar com `axis=0` produz colunas que somam 1. Para classificação por exemplo,
o invariante relevante é a soma de cada linha.


In [ ]:
def wrong_softmax_axis_zero(logits):
    logits = validate_logits(logits)
    shifted = logits - np.max(logits, axis=0, keepdims=True)
    exp_shifted = np.exp(shifted)
    return exp_shifted / exp_shifted.sum(axis=0, keepdims=True)


wrong_probabilities = wrong_softmax_axis_zero(batch_logits)
wrong_row_sums = wrong_probabilities.sum(axis=1)
wrong_column_sums = wrong_probabilities.sum(axis=0)

print("somas por linha com eixo errado:", np.round(wrong_row_sums, 6))
print("somas por coluna com eixo errado:", np.round(wrong_column_sums, 6))


### 11. Multiclasse versus multirrótulo

Softmax força competição. Sigmoides independentes podem atribuir alta probabilidade a
várias classes, como exige um problema multirrótulo.


In [ ]:
def sigmoid_stable(values):
    values = np.asarray(values, dtype=DTYPE)
    result = np.empty_like(values)
    positive = values >= 0.0
    result[positive] = 1.0 / (1.0 + np.exp(-values[positive]))
    exp_values = np.exp(values[~positive])
    result[~positive] = exp_values / (1.0 + exp_values)
    return result


shared_logits = np.array([[2.0, 2.0, -2.0]])
exclusive = softmax(shared_logits)
independent = sigmoid_stable(shared_logits)

print("softmax:", np.round(exclusive, 6), "soma=", exclusive.sum())
print("sigmoides:", np.round(independent, 6), "soma=", independent.sum())


### 12. Gradiente local e diferenças centrais

Para a redução média,

$$
\nabla_{\mathbf Z}L=\frac{\mathbf P-\mathbf Q}{m}.
$$

Comparamos cada coordenada a uma diferença central. Este teste cobre apenas a operação
softmax–cross-entropy, não o backward das camadas anteriores.


In [ ]:
def cross_entropy_gradient_dense(logits, targets, reduction="mean"):
    logits = validate_logits(logits)
    targets = validate_dense_targets(targets, logits.shape)
    gradient = softmax(logits) - targets
    if reduction == "mean":
        gradient = gradient / logits.shape[0]
    elif reduction == "sum":
        pass
    else:
        raise MulticlassContractError("gradiente exige reduction 'sum' ou 'mean'")
    return gradient


def central_difference_gradient(logits, targets, epsilon=1e-5):
    numerical = np.zeros_like(logits, dtype=DTYPE)
    for index in np.ndindex(logits.shape):
        plus = logits.copy()
        minus = logits.copy()
        plus[index] += epsilon
        minus[index] -= epsilon
        numerical[index] = (
            cross_entropy_dense(plus, targets, "mean")
            - cross_entropy_dense(minus, targets, "mean")
        ) / (2.0 * epsilon)
    return numerical


gradient_logits = rng.normal(0.0, 1.0, size=(4, 5))
gradient_labels = np.array([0, 3, 1, 4])
gradient_targets = one_hot(gradient_labels, class_count=5)
analytical_gradient = cross_entropy_gradient_dense(
    gradient_logits, gradient_targets, "mean"
)
numerical_gradient = central_difference_gradient(
    gradient_logits, gradient_targets
)
gradient_max_error = float(np.max(np.abs(
    analytical_gradient - numerical_gradient
)))
gradient_row_sum_error = float(np.max(np.abs(
    analytical_gradient.sum(axis=1)
)))

print({
    "erro_máximo_diferenças_centrais": gradient_max_error,
    "erro_soma_das_linhas": gradient_row_sum_error,
})


### 13. Curva da loss conforme a margem correta

Fixamos três classes com logits `[margem, 0, 0]` e alvo 0. A loss é
$\log(1+2e^{-\text{margem}})$: cai quando a classe correta se separa das demais.


In [ ]:
margin_grid = np.linspace(-10.0, 10.0, 401)
curve_logits = np.column_stack([
    margin_grid,
    np.zeros_like(margin_grid),
    np.zeros_like(margin_grid),
])
curve_labels = np.zeros(margin_grid.size, dtype=np.int64)
curve_losses = cross_entropy_sparse(curve_logits, curve_labels, "none")
curve_probabilities = softmax(curve_logits)[:, 0]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(margin_grid, curve_probabilities, color="#1b9e77", linewidth=2)
axes[0].axhline(1 / 3, color="gray", linestyle="--", label="uniforme: 1/3")
axes[0].set(title="Probabilidade da classe correta",
            xlabel="margem do logit correto", ylabel="probabilidade")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(margin_grid, curve_losses, color="#d95f02", linewidth=2)
axes[1].axhline(np.log(3), color="gray", linestyle="--", label="baseline: log(3)")
axes[1].set(title="Cross-entropy da classe correta",
            xlabel="margem do logit correto", ylabel="loss")
axes[1].legend()
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

print("Texto alternativo: ao aumentar a margem correta, a probabilidade sobe de "
      "quase zero para um e a cross-entropy cai de aproximadamente dez para zero.")


## Checks

Os contratos abaixo cobrem valores manuais, domínios, shapes, estabilidade, eixo,
reduções, invariâncias, semântica multirrótulo e gradiente local.


In [ ]:
assert np.allclose(manual_probabilities, manual_expected, atol=1e-15, rtol=0.0)
assert np.allclose(manual_probabilities.sum(axis=1), 1.0, atol=1e-15)
assert np.isclose(manual_loss_class_0, 0.4076059644443806, atol=1e-15)
assert np.isclose(manual_loss_class_2, 2.4076059644443806, atol=1e-15)
assert one_hot_equivalence_error < 5e-16

assert np.allclose(uniform_none, np.log(class_count), atol=1e-15)
assert np.isclose(uniform_mean, np.log(class_count), atol=1e-15)
assert np.isclose(uniform_sum, uniform_mean * uniform_none.size, atol=1e-14)
assert probability_shift_error < 4e-11
assert loss_shift_error < 4e-11
assert permutation_error < 5e-16

assert not np.isfinite(naive_probabilities).all()
assert not np.isfinite(naive_losses).all()
assert np.isfinite(stable_probabilities).all()
assert np.isfinite(stable_losses).all()
assert np.allclose(stable_losses, np.array([0.0, 2000.0]))

assert not np.allclose(wrong_row_sums, 1.0)
assert np.allclose(wrong_column_sums, 1.0)
assert np.isclose(exclusive.sum(), 1.0)
assert independent.sum() > 1.7

assert gradient_max_error < 3e-11
assert gradient_row_sum_error < 1e-16

try:
    cross_entropy_sparse(np.zeros((2, 3)), np.array([[0], [1]]))
except MulticlassContractError:
    pass
else:
    raise AssertionError("label esparso (m, 1) deveria ser rejeitado")

try:
    cross_entropy_sparse(np.zeros((2, 3)), np.array([0.0, 1.0]))
except MulticlassContractError:
    pass
else:
    raise AssertionError("label esparso float deveria ser rejeitado")

try:
    cross_entropy_dense(np.zeros((1, 3)), np.array([[0.8, 0.4, 0.0]]))
except MulticlassContractError:
    pass
else:
    raise AssertionError("alvo denso com soma 1.2 deveria ser rejeitado")

print("25 contratos verificados com sucesso.")


## Resultados confirmados

- Softmax de `[2, 1, 0]`: `[0.665241, 0.244728, 0.090031]`.
- Cross-entropy com alvo 0: `0.407605964`; com alvo 2: `2.407605964`.
- Loss uniforme com cinco classes: `log(5) = 1.609437912`.
- Forma estável em `[1000, 0, -1000]`: losses `0` e `2000`; forma ingênua não finita.
- Rótulos esparsos e one-hot coincidiram até precisão de máquina.
- Gradiente $\mathbf{p}-\mathbf{q}$: erro máximo por diferenças centrais abaixo de `3e-11`.


## Next Steps

Na Aula 08, formalizaremos operações como nós de um grafo computacional. Cada nó terá
`forward` e `backward`, e o gradiente local será combinado com um gradiente upstream
por *vector-Jacobian products*, sem materializar Jacobianas gigantes.
